# RAG Eval Lab — Schema & Data Model

Medallion architecture for multi-hop QA evaluation over HotPotQA. Separates ground truth (bronze/silver) from experiment tracking (exp layer), so every retrieval run, generation, and judge verdict traces back to the exact config that produced it.

## Full 12-Table Schema

### Bronze Layer — raw ingest
| Table | Columns | Grain |
|---|---|---|
| `bronze_hotpot_raw` | `id` · `record` (VARIANT/JSON) · `ingest_ts` | 1 row per raw HotPotQA record |

### Silver Layer — normalized ground truth
| Table | Columns | Grain |
|---|---|---|
| `silver_questions` | `q_id` · `question` · `answer` · `type` · `level` | 1 row per question |
| `silver_paragraphs` | `para_id` · `title` · `sentences` (ARRAY\<STRING\>) · `text` | 1 row per unique Wikipedia paragraph |
| `silver_qrels` | `q_id` · `para_id` · `sent_id` | 1 row per (question, gold paragraph, supporting sentence) |

### Experiment Layer — Day 1: Retrieval
| Table | Columns | Grain |
|---|---|---|
| `exp_chunk_configs` | `chunk_cfg_id` · `sents_per_chunk` · `overlap` | 1 row per chunking parameter set |
| `silver_chunks` | `chunk_id` · `chunk_cfg_id` · `para_id` · `sent_start` · `sent_end` · `text` · `embedding` · `embed_model` | 1 row per chunk per config per paragraph |
| `exp_configs` | `cfg_id` · `chunk_cfg_id` · `strategy` (dense/bm25/hybrid) | 1 row per experiment definition |
| `exp_retrievals` | `cfg_id` · `q_id` · `rank` · `chunk_id` · `score` | 1 row per retrieved chunk per question per config |
| `exp_metrics` | `cfg_id` · `k` · `recall` · `mrr` · `ndcg` | 1 row per config per cutoff k |

### Experiment Layer — Day 2: Generation
| Table | Columns | Grain |
|---|---|---|
| `exp_generations` | `gen_id` · `cfg_id` · `q_id` · `answer` · `trace_id` · `latency_ms` · `tokens_in` · `tokens_out` | 1 row per generated answer per question per config |

### Experiment Layer — Day 3: Judging
| Table | Columns | Grain |
|---|---|---|
| `exp_judges` | `judge_id` · `kind` · `model` · `prompt_version` | 1 row per judge configuration |
| `exp_judgments` | `gen_id` · `judge_id` · `verdict` · `confidence` · `cost_usd` | 1 row per judgment per generation per judge |

In [0]:
%pip install --upgrade "mlflow[databricks]>=3.4.0" openai datasets rank_bm25 "fsspec>=2024.2.0" huggingface_hub
dbutils.library.restartPython()

In [0]:
import json
from datetime import datetime
from datasets import load_dataset
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

# Load full validation split from HuggingFace
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
ingest_ts = datetime.utcnow()

# Bronze: freeze the raw JSON record with an ingest timestamp
bronze_rows = [
    (r["id"], json.dumps(r, ensure_ascii=False), ingest_ts)
    for r in ds
]
bronze_schema = StructType([
    StructField("id", StringType()),
    StructField("record", StringType()),      # raw JSON blob
    StructField("ingest_ts", TimestampType()),
])
df_bronze = spark.createDataFrame(bronze_rows, schema=bronze_schema)
df_bronze.createOrReplaceTempView("bronze_hotpot_raw")

print(f"bronze_hotpot_raw: {df_bronze.count():,} rows")
df_bronze.show(3, truncate=80)

In [0]:
import json
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, ArrayType
)
from pyspark.sql import functions as F

# Parse bronze rows back into Python dicts
raw_rows = spark.table("bronze_hotpot_raw").select("record").collect()
records = [json.loads(row.record) for row in raw_rows]

# ── silver_questions ────────────────────────────────────────────────
q_rows = [
    (r["id"], r["question"], r["answer"], r["type"], r["level"])
    for r in records
]
q_schema = StructType([
    StructField("q_id", StringType()),
    StructField("question", StringType()),
    StructField("answer", StringType()),
    StructField("type", StringType()),
    StructField("level", StringType()),
])
df_questions = spark.createDataFrame(q_rows, schema=q_schema)
df_questions.createOrReplaceTempView("silver_questions")

# ── silver_paragraphs  (deduplicated by title) ─────────────────────
# A paragraph may appear across multiple questions; store it once.
para_map = {}  # title -> sentences list
for r in records:
    for title, sents in zip(r["context"]["title"], r["context"]["sentences"]):
        if title not in para_map:
            para_map[title] = [s.strip() for s in sents]

para_rows = [
    (f"p_{i}", title, sents, " ".join(sents))
    for i, (title, sents) in enumerate(para_map.items())
]
para_schema = StructType([
    StructField("para_id", StringType()),
    StructField("title", StringType()),
    StructField("sentences", ArrayType(StringType())),
    StructField("text", StringType()),
])
df_paragraphs = spark.createDataFrame(para_rows, schema=para_schema)
df_paragraphs.createOrReplaceTempView("silver_paragraphs")

# Build a title -> para_id lookup for qrels
title_to_para_id = {title: f"p_{i}" for i, (title, _) in enumerate(para_map.items())}

# ── silver_qrels  (question × gold paragraph × supporting sentence) ─
qrel_rows = []
for r in records:
    for sf_title, sf_sid in zip(
        r["supporting_facts"]["title"], r["supporting_facts"]["sent_id"]
    ):
        para_id = title_to_para_id.get(sf_title)
        if para_id:
            qrel_rows.append((r["id"], para_id, sf_sid))

qrel_schema = StructType([
    StructField("q_id", StringType()),
    StructField("para_id", StringType()),
    StructField("sent_id", IntegerType()),
])
df_qrels = spark.createDataFrame(qrel_rows, schema=qrel_schema)
df_qrels.createOrReplaceTempView("silver_qrels")

print(f"silver_questions  : {df_questions.count():,} rows")
print(f"silver_paragraphs : {df_paragraphs.count():,} rows (deduplicated by title)")
print(f"silver_qrels      : {df_qrels.count():,} rows")

df_questions.show(3, truncate=60)
df_paragraphs.select("para_id", "title", F.size("sentences").alias("n_sents")).show(3, truncate=50)
df_qrels.show(5, truncate=40)

In [0]:
import re, hashlib, numpy as np
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, ArrayType, FloatType
)
from databricks.sdk import WorkspaceClient

# ── exp_chunk_configs ──────────────────────────────────────────────
chunk_configs = [
    ("cfg_3_1", 3, 1),   # 3 sentences per chunk, 1 sentence overlap
    ("cfg_5_2", 5, 2),   # 5 sentences per chunk, 2 sentence overlap
]
cfg_schema = StructType([
    StructField("chunk_cfg_id", StringType()),
    StructField("sents_per_chunk", IntegerType()),
    StructField("overlap", IntegerType()),
])
df_chunk_configs = spark.createDataFrame(chunk_configs, schema=cfg_schema)
df_chunk_configs.createOrReplaceTempView("exp_chunk_configs")

# ── Build chunks from silver_paragraphs ────────────────────────────
para_data = df_paragraphs.select("para_id", "sentences").collect()

def make_chunks(sentences, sents_per_chunk, overlap):
    """Sliding window chunker: yields (sent_start, sent_end, text)."""
    step = max(sents_per_chunk - overlap, 1)
    for start in range(0, len(sentences), step):
        end = min(start + sents_per_chunk, len(sentences))
        yield start, end, " ".join(sentences[start:end])
        if end >= len(sentences):
            break

chunk_rows = []
for para_row in para_data:
    sents = para_row.sentences
    for cfg_id, spc, ovl in chunk_configs:
        for s_start, s_end, text in make_chunks(sents, spc, ovl):
            cid = hashlib.md5(f"{para_row.para_id}_{cfg_id}_{s_start}_{s_end}".encode()).hexdigest()[:12]
            chunk_rows.append((cid, cfg_id, para_row.para_id, s_start, s_end, text))

print(f"Total chunks generated: {len(chunk_rows):,}")

# ── Embed chunks (batch, using Databricks embedding endpoint) ──────
EMB_MODEL = "databricks-gte-large-en"
client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Batch embed in groups of 256
texts = [r[5] for r in chunk_rows]
all_embeddings = []
BATCH = 150
for i in range(0, len(texts), BATCH):
    batch = texts[i:i+BATCH]
    resp = client.embeddings.create(model=EMB_MODEL, input=batch)
    all_embeddings.extend([d.embedding for d in resp.data])
    if (i // BATCH) % 20 == 0:
        print(f"  embedded {min(i+BATCH, len(texts)):,} / {len(texts):,}")

print(f"  done — {len(all_embeddings):,} embeddings")

# ── Build silver_chunks DataFrame ──────────────────────────────────
chunk_rows_full = [
    (r[0], r[1], r[2], r[3], r[4], r[5], all_embeddings[i], EMB_MODEL)
    for i, r in enumerate(chunk_rows)
]
chunk_schema = StructType([
    StructField("chunk_id", StringType()),
    StructField("chunk_cfg_id", StringType()),
    StructField("para_id", StringType()),
    StructField("sent_start", IntegerType()),
    StructField("sent_end", IntegerType()),
    StructField("text", StringType()),
    StructField("embedding", ArrayType(FloatType())),
    StructField("embed_model", StringType()),
])
df_chunks = spark.createDataFrame(chunk_rows_full, schema=chunk_schema)
df_chunks.createOrReplaceTempView("silver_chunks")

print(f"\nsilver_chunks: {df_chunks.count():,} rows")
df_chunks.select("chunk_id", "chunk_cfg_id", "para_id", "sent_start", "sent_end", "embed_model").show(5, truncate=30)

In [0]:
import numpy as np
from rank_bm25 import BM25Okapi
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType
)

# ── exp_configs ────────────────────────────────────────────────────
# Cross: 2 chunk configs × 2 strategies = 4 experiment configs
exp_cfg_rows = []
for cfg_id_suffix, chunk_cfg in [("3_1", "cfg_3_1"), ("5_2", "cfg_5_2")]:
    for strategy in ["bm25", "dense"]:
        exp_cfg_rows.append((f"exp_{strategy}_{cfg_id_suffix}", chunk_cfg, strategy))

exp_cfg_schema = StructType([
    StructField("cfg_id", StringType()),
    StructField("chunk_cfg_id", StringType()),
    StructField("strategy", StringType()),
])
df_exp_configs = spark.createDataFrame(exp_cfg_rows, schema=exp_cfg_schema)
df_exp_configs.createOrReplaceTempView("exp_configs")
print("exp_configs:")
df_exp_configs.show()

# ── Retrieval: run BM25 and dense on a sample of 50 questions ──────
import re

question_rows = df_questions.limit(50).collect()
chunk_local = df_chunks.select("chunk_id", "chunk_cfg_id", "text", "embedding").collect()

# Pre-index chunks by chunk_cfg_id
chunks_by_cfg = {}
for c in chunk_local:
    chunks_by_cfg.setdefault(c.chunk_cfg_id, []).append(c)

tok = lambda t: re.findall(r"[a-z0-9]+", t.lower())
K = 10

retrieval_rows = []
for exp_cfg in exp_cfg_rows:
    cfg_id, chunk_cfg, strategy = exp_cfg
    chunks = chunks_by_cfg[chunk_cfg]
    chunk_texts = [c.text for c in chunks]
    chunk_ids = [c.chunk_id for c in chunks]

    if strategy == "bm25":
        tokenized = [tok(t) for t in chunk_texts]
        bm25 = BM25Okapi(tokenized)

    if strategy == "dense":
        emb_matrix = np.array([c.embedding for c in chunks])
        norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
        unit_embs = emb_matrix / norms

    for q_row in question_rows:
        q_id = q_row.q_id
        question = q_row.question

        if strategy == "bm25":
            scores = bm25.get_scores(tok(question))
        else:  # dense
            q_emb = np.array(client.embeddings.create(
                model=EMB_MODEL, input=[question]
            ).data[0].embedding)
            q_unit = q_emb / np.linalg.norm(q_emb)
            scores = unit_embs @ q_unit

        top_k_idx = np.argsort(-scores)[:K]
        for rank, idx in enumerate(top_k_idx, 1):
            retrieval_rows.append((cfg_id, q_id, rank, chunk_ids[idx], float(scores[idx])))

    print(f"  {cfg_id}: {len(question_rows)} questions × top-{K} done")

ret_schema = StructType([
    StructField("cfg_id", StringType()),
    StructField("q_id", StringType()),
    StructField("rank", IntegerType()),
    StructField("chunk_id", StringType()),
    StructField("score", FloatType()),
])
df_retrievals = spark.createDataFrame(retrieval_rows, schema=ret_schema)
df_retrievals.createOrReplaceTempView("exp_retrievals")
print(f"\nexp_retrievals: {df_retrievals.count():,} rows")
df_retrievals.show(5, truncate=30)

In [0]:
import math
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# ── Compute metrics from exp_retrievals + silver_qrels + silver_chunks ──
# A retrieved chunk is "relevant" if its paragraph + sentence range
# overlaps any gold (q_id, para_id, sent_id) in silver_qrels.

retrieval_local = df_retrievals.collect()
chunk_meta = {c.chunk_id: c for c in df_chunks.select("chunk_id", "para_id", "sent_start", "sent_end").collect()}
qrel_local = df_qrels.collect()

# Build gold set: q_id -> set of (para_id, sent_id)
gold = {}
for qr in qrel_local:
    gold.setdefault(qr.q_id, set()).add((qr.para_id, qr.sent_id))

def is_relevant(q_id, chunk_id):
    """Check if chunk covers any gold sentence for this question."""
    cm = chunk_meta.get(chunk_id)
    if not cm:
        return False
    gold_set = gold.get(q_id, set())
    for para_id, sent_id in gold_set:
        if cm.para_id == para_id and cm.sent_start <= sent_id < cm.sent_end:
            return True
    return False

# Group retrievals by (cfg_id, q_id), sorted by rank
from collections import defaultdict
grouped = defaultdict(list)
for r in retrieval_local:
    grouped[(r.cfg_id, r.q_id)].append((r.rank, r.chunk_id))

for key in grouped:
    grouped[key].sort()

def compute_metrics_at_k(grouped, k):
    """Compute recall@k, MRR@k, NDCG@k per config."""
    cfg_metrics = defaultdict(lambda: {"recalls": [], "rrs": [], "dcgs": []})

    for (cfg_id, q_id), ranked in grouped.items():
        gold_count = len(gold.get(q_id, set()))
        if gold_count == 0:
            continue

        hits = 0
        rr = 0.0
        dcg = 0.0
        for rank, chunk_id in ranked[:k]:
            rel = 1 if is_relevant(q_id, chunk_id) else 0
            hits += rel
            if rel and rr == 0:
                rr = 1.0 / rank
            dcg += rel / math.log2(rank + 1)

        # Ideal DCG: all relevant docs at top ranks
        idcg = sum(1.0 / math.log2(i + 2) for i in range(min(gold_count, k)))
        ndcg = dcg / idcg if idcg > 0 else 0.0

        m = cfg_metrics[cfg_id]
        m["recalls"].append(hits / gold_count)
        m["rrs"].append(rr)
        m["dcgs"].append(ndcg)

    return cfg_metrics

metric_rows = []
for k in [1, 3, 5, 10]:
    cfg_metrics = compute_metrics_at_k(grouped, k)
    for cfg_id, m in cfg_metrics.items():
        metric_rows.append((
            cfg_id, k,
            round(sum(m["recalls"]) / len(m["recalls"]), 4),
            round(sum(m["rrs"]) / len(m["rrs"]), 4),
            round(sum(m["dcgs"]) / len(m["dcgs"]), 4),
        ))

metric_schema = StructType([
    StructField("cfg_id", StringType()),
    StructField("k", IntegerType()),
    StructField("recall", DoubleType()),
    StructField("mrr", DoubleType()),
    StructField("ndcg", DoubleType()),
])
df_metrics = spark.createDataFrame(metric_rows, schema=metric_schema)
df_metrics.createOrReplaceTempView("exp_metrics")

print("exp_metrics:")
display(df_metrics.orderBy("cfg_id", "k"))

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, FloatType
)

# ── Day 2: exp_generations (schema only, populated later) ──────────
gen_schema = StructType([
    StructField("gen_id", StringType()),
    StructField("cfg_id", StringType()),
    StructField("q_id", StringType()),
    StructField("answer", StringType()),
    StructField("trace_id", StringType()),
    StructField("latency_ms", FloatType()),
    StructField("tokens_in", IntegerType()),
    StructField("tokens_out", IntegerType()),
])
df_generations = spark.createDataFrame([], schema=gen_schema)
df_generations.createOrReplaceTempView("exp_generations")

# ── Day 3: exp_judges (schema only) ────────────────────────────────
judge_schema = StructType([
    StructField("judge_id", StringType()),
    StructField("kind", StringType()),        # groundedness, relevance, faithfulness
    StructField("model", StringType()),        # ai_classify, gpt-4, etc.
    StructField("prompt_version", StringType()),
])
df_judges = spark.createDataFrame([], schema=judge_schema)
df_judges.createOrReplaceTempView("exp_judges")

# ── Day 3: exp_judgments (schema only) ─────────────────────────────
judgment_schema = StructType([
    StructField("gen_id", StringType()),
    StructField("judge_id", StringType()),
    StructField("verdict", StringType()),
    StructField("confidence", DoubleType()),
    StructField("cost_usd", DoubleType()),
])
df_judgments = spark.createDataFrame([], schema=judgment_schema)
df_judgments.createOrReplaceTempView("exp_judgments")

print("Day 2/3 stub tables registered (empty, schema only):")
print(f"  exp_generations : {df_generations.count()} rows — schema: {[f.name for f in gen_schema]}")
print(f"  exp_judges      : {df_judges.count()} rows — schema: {[f.name for f in judge_schema]}")
print(f"  exp_judgments   : {df_judgments.count()} rows — schema: {[f.name for f in judgment_schema]}")

In [0]:
# ── Final summary: all 12 tables ───────────────────────────────────
tables = [
    ("bronze_hotpot_raw",          "bronze"),
    ("silver_questions",           "silver"),
    ("silver_paragraphs",          "silver"),
    ("silver_qrels",               "silver"),
    ("exp_chunk_configs",          "experiment"),
    ("silver_chunks",              "experiment"),
    ("exp_configs",                "experiment"),
    ("exp_retrievals",             "experiment"),
    ("exp_metrics",                "experiment"),
    ("exp_generations",            "day2 stub"),
    ("exp_judges",                 "day3 stub"),
    ("exp_judgments",              "day3 stub"),
]

print(f"{'Table':<28} {'Layer':<12} {'Rows':>8}")
print("─" * 50)
for name, layer in tables:
    count = spark.table(name).count()
    print(f"{name:<28} {layer:<12} {count:>8,}")